# Práctica de detección de objetos — Visión por computador 2026-2027

### **Grupo:** N — **Integrantes:** Lucas Amado Oseira, María Lestón Geada y _(⚠️ falta el tercer integrante: el guion exige equipos de exactamente 3)_
##### **Dataset:** Lettuce Pallets (`Francesco/lettuce-pallets`, Roboflow-100)
##### **Detectores:** Cascade R-CNN (Detectron2, two-stage) y SSD300-VGG16 (Torchvision, one-stage), ambos preentrenados en MS COCO

## Cómo usar este cuaderno y resumen de decisiones

**Orden de trabajo (se ejecuta de arriba abajo, en Colab con GPU T4):**
1. Instalar dependencias → 2. Configurar → 3. Cargar y limpiar los datos → 4. Analizar el dataset →
5. Baseline SSD300 → 6. Baseline Cascade R-CNN → 7. Mejoras (E1–E6) → 8. Resultados y análisis de fallos.

Cada entrenamiento guarda su checkpoint y sus predicciones en Drive: si Colab se desconecta, **se vuelve a ejecutar el cuaderno entero y lo ya entrenado se reutiliza**
(solo se recalcula la evaluación). Los resultados de todos los experimentos se acumulan en `results.csv`.

**Resumen de decisiones (cada una se justifica en su sección):**

| Decisión | Elección | Motivo breve |
|---|---|---|
| Clases | Se elimina `lettuce` (id 0) → 5 clases | Es la categoría "padre" de Roboflow, sin anotaciones; el guion habla de 5 clases |
| Protocolo | Entrenar en *train*, elegir en *val*, reportar *test* | Evita sobreajustar las decisiones al test |
| Baseline | Fine-tuning de pesos COCO, **sin aumento de datos**, un solo LR | Punto de partida limpio: cada mejora cambia **una** cosa |
| Evaluación | Mismo código (`evaluate_coco`) y mismos umbrales para ambos detectores | Comparación justa SSD vs Cascade |
| Mejoras | Aumento de datos, LR diferenciado, combinación, TTA, ensemble | Cada una responde a una característica concreta del dataset |

## 0. Instalación del entorno

Ejecutamos esta celda **una vez** por sesión nueva de Colab con GPU T4. Detectron2 se compila desde el repositorio oficial (tarda varios minutos).
Si Colab pide reiniciar la sesión, reiniciamos y continuamos desde la celda de imports.

In [ ]:
# Dependencias de evaluación y ensembles
!pip -q install pycocotools ensemble-boxes

# Detectron2 se compila desde el repositorio (tarda varios minutos)
!pip -q install 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation

## 1. Imports y configuración

In [ ]:
import os, sys, io, json, time, math, copy, random, tarfile, contextlib, gc, importlib.util
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Importamos las librerías de pycocotools para la evaluación del modelo y el manejo del dataset COCO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Importamos las librerías de Detectron2 para la configuración, entrenamiento y evaluación del modelo
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.evaluation import COCOEvaluator
from detectron2.data import (DatasetCatalog, MetadataCatalog, DatasetMapper,
                                build_detection_train_loader, transforms as T)
from detectron2.data.datasets import register_coco_instances

**Decisiones de configuración y por qué:**
- **`SEED = 2710`** (+ cuDNN determinista): el guion pide reproducibilidad.
- **`SCORE_THRESH_EVAL = 0.001`**: el mAP integra toda la curva Precisión-Recall; si se filtran las detecciones poco confiables se
  recorta artificialmente la curva. Se usa un umbral muy bajo para evaluar (para *visualizar* se usará uno mayor).
- **`NMS_IOU = 0.5`** y **`MAX_DETS = 300`** iguales en ambos detectores: el dataset tiene ~13 objetos por imagen de media
  (13.812 cajas en 1.060 imágenes) y habrá imágenes más densas; el límite por defecto de 100 detecciones podría recortarlas.
- Todos estos parámetros de postprocesado **son comunes a SSD y Cascade**, para que la diferencia de resultados sea del modelo y no del postprocesado.

In [ ]:
SEED = 2710
DATASET_NAME = "lettuce-pallets"

DRIVE_TAR = "/content/drive/MyDrive/vision_p6/dataset.tar.gz"      # el tar.gz ya subido a Drive
USE_DRIVE = True                                                    # guardar checkpoints/resultados en Drive
DRIVE_WORK_DIR = "/content/drive/MyDrive/vision_p6/work"
LOCAL_WORK_DIR = "work"

PRETRAINED = True               # pesos COCO (obligatorio en la práctica)

# Parámetros de inferencia COMUNES a ambos detectores (ver justificación arriba)
SCORE_THRESH_EVAL = 0.001
NMS_IOU = 0.5
MAX_DETS = 300

SPLITS = ("train", "val", "test")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
def seed_everything(seed=SEED, deterministic=True):
    """Fija todas las semillas. 'deterministic' hace cuDNN determinista (algo más lento)."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    """Semilla para los workers del DataLoader (si no, el aumento de datos no es reproducible)."""
    np.random.seed(SEED + worker_id); random.seed(SEED + worker_id)

def make_generator(seed=SEED):
    return torch.Generator().manual_seed(seed)

seed_everything()

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path(DRIVE_WORK_DIR)
else:
    WORK_DIR = Path(LOCAL_WORK_DIR)
for sub in ("ckpt", "preds", "figs", "d2"):
    (WORK_DIR / sub).mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)   # disco local (rápido)
CLEAN_DIR = DATA_DIR / "clean"; CLEAN_DIR.mkdir(exist_ok=True)
print("WORK_DIR:", WORK_DIR.resolve())

## 2. Datos: extracción, limpieza y formato unificado

**Paso:** extraer el `dataset.tar.gz` (formato COCO de Roboflow-100) y localizar el JSON y la carpeta de imágenes de cada split.

**Decisión:** localizar los splits por el **nombre exacto de la carpeta** (`train`, `valid`, `test`) y no por subcadenas de la ruta como antes.
**Por qué:** buscar "val" o "test" en toda la ruta puede dar falsos positivos (p. ej. una carpeta que contenga "validation" o "latest"); el nombre exacto es robusto.

In [ ]:
SPLIT_DIR_ALIASES = {"train": "train", "valid": "val", "validation": "val", "val": "val", "test": "test"}

def extract_tar(tar_path, dest):
    """Extrae el tar.gz si el destino está vacío (en Colab /content se borra al cerrar la sesión)."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    if not any(dest.iterdir()):
        with tarfile.open(tar_path, "r:gz") as tar:
            try: tar.extractall(dest, filter="data")
            except TypeError: tar.extractall(dest)
    return dest

def locate_raw_splits(root):
    raw = {}
    for js in Path(root).rglob("*.json"):
        key = SPLIT_DIR_ALIASES.get(js.parent.name.lower())
        if key and key not in raw:
            raw[key] = {"json": js, "images": js.parent} # en Roboflow, las imágenes están junto al JSON
    missing = [s for s in SPLITS if s not in raw]
    assert not missing, f"Faltan splits: {missing}"
    return raw

RAW = locate_raw_splits(extract_tar(DRIVE_TAR, DATA_DIR / "raw"))
for s in SPLITS: print(f"{s:5s} -> {RAW[s]['json']}")

**Paso:** limpiar las anotaciones y crear un JSON limpio por split.

**Decisiones y justificación:**
1. **Eliminar la categoría `lettuce` (id 0).** En Roboflow es la categoría "padre" del dataset y no tiene anotaciones propias. Si se dejara, el modelo
   tendría 6 clases en vez de las 5 reales del guion, con una clase "fantasma" que nunca aparece.
2. **Renumerar las clases a 1…5.** En torchvision la etiqueta 0 es el fondo; con ids contiguos 1…5 la etiqueta del modelo **coincide** con el `category_id`
   y no hace falta ninguna conversión (la versión anterior restaba 1 al evaluar y desplazaba todas las clases).
3. **Recortar las cajas a la imagen y descartar las degeneradas** (ancho o alto < 1 px): una caja de tamaño nulo provoca errores y pérdidas `NaN` al entrenar.
4. **Se conservan las imágenes sin objetos** (en el JSON): se usan en la evaluación; Detectron2 las ignora solo al entrenar.

In [ ]:
def build_clean_coco(raw, out_dir):
    """Genera un JSON COCO limpio por split. Devuelve (CLEAN, class_names, class_ids)."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    train = json.loads(Path(raw["train"]["json"]).read_text())
    old_names = {c["id"]: c["name"] for c in train["categories"]}
    used_ids = sorted({a["category_id"] for a in train["annotations"]})
    dropped = [n for i, n in old_names.items() if i not in used_ids]
    print("Categorías descartadas (sin anotaciones en train):", dropped)
    name2new = {old_names[i]: n + 1 for n, i in enumerate(used_ids)}     # ids nuevos 1..K

    clean = {}
    for split in SPLITS:
        d = json.loads(Path(raw[split]["json"]).read_text())
        id2name = {c["id"]: c["name"] for c in d["categories"]}
        imgs = {im["id"]: im for im in d["images"]}
        new_anns, stats = [], Counter()
        for a in d["annotations"]:
            name = id2name.get(a["category_id"])
            if name not in name2new:
                stats["clase_descartada"] += 1; continue
            im = imgs[a["image_id"]]; x, y, w, h = a["bbox"]
            x1, y1 = max(0.0, x), max(0.0, y)
            x2, y2 = min(float(im["width"]), x + w), min(float(im["height"]), y + h)
            if x2 - x1 < 1 or y2 - y1 < 1:
                stats["caja_degenerada"] += 1; continue
            new_anns.append({"id": len(new_anns) + 1, "image_id": a["image_id"],
                             "category_id": name2new[name], "bbox": [x1, y1, x2 - x1, y2 - y1],
                             "area": (x2 - x1) * (y2 - y1), "iscrowd": 0})
        out = {"images": d["images"], "annotations": new_anns,
               "categories": [{"id": i, "name": n} for n, i in name2new.items()]}
        p = out_dir / f"{split}_clean.json"; p.write_text(json.dumps(out))
        clean[split] = {"json": p, "images": Path(raw[split]["images"])}
        print(f"{split:5s}: {len(d['images'])} imgs | {len(d['annotations'])} -> {len(new_anns)} anotaciones | descartes: {dict(stats)}")
    names = list(name2new.keys()); ids = list(name2new.values())
    return clean, names, ids

CLEAN, CLASS_NAMES, CLASS_IDS = build_clean_coco(RAW, CLEAN_DIR)
K = len(CLASS_NAMES)                      # nº de clases reales
CID2IDX = {c: i for i, c in enumerate(CLASS_IDS)}
print(f"\nK = {K} clases:", dict(zip(CLASS_IDS, CLASS_NAMES)))
assert K == 5, "El guion dice 5 clases para lettuce-pallets: revisad la limpieza"

COCO_GT = {s: COCO(str(CLEAN[s]["json"])) for s in SPLITS}

## 3. Análisis exploratorio del dataset

**Paso:** medir las características que condicionan el diseño de las mejoras: reparto de clases, nº de objetos por imagen y **tamaño de los objetos**.
Es lo que pide el guion en la exposición (punto 1) y lo que **justifica** las decisiones de la fase de mejora.

**Qué buscamos y por qué importa:**
- *Desbalance de clases* → si hay clases raras, plantear muestreo balanceado o ponderar la pérdida.
- *Tamaño de objeto* → SSD300 reescala **todas** las imágenes a 300×300; los objetos pequeños pierden detalle. Calculamos el tamaño efectivo tras el reescalado.
- *Vista cenital* → no hay "arriba/abajo" privilegiado: los giros y volteos de la imagen producen escenas igual de realistas (justifica el aumento geométrico).

In [ ]:
def annotations_dataframe(coco, split):
    rows = []
    for a in coco.anns.values():
        im = coco.imgs[a["image_id"]]; x, y, w, h = a["bbox"]
        area = w * h
        rows.append({"split": split, "image_id": a["image_id"], "cls": CLASS_NAMES[CID2IDX[a["category_id"]]],
                     "w": w, "h": h, "area": area, "sqrt_area": math.sqrt(area),
                     "rel_size": math.sqrt(area / (im["width"] * im["height"])), "aspect": w / h})
    df = pd.DataFrame(rows)
    # categorías de tamaño de COCO (en píxeles de la imagen original)
    df["size_cat"] = pd.cut(df["area"], [0, 32**2, 96**2, np.inf], labels=["small", "medium", "large"])
    return df

ANN_DF = pd.concat([annotations_dataframe(COCO_GT[s], s) for s in SPLITS], ignore_index=True)

print("== Imágenes y anotaciones por split ==")
summary = []
for s in SPLITS:
    c = COCO_GT[s]; per_img = Counter(a["image_id"] for a in c.anns.values())
    n_img = len(c.imgs); n_empty = sum(1 for i in c.imgs if per_img.get(i, 0) == 0)
    summary.append({"split": s, "imágenes": n_img, "anotaciones": len(c.anns), "imgs_sin_objetos": n_empty,
                    "objs/img media": round(len(c.anns) / n_img, 1), "objs/img máx": max(per_img.values())})
display(pd.DataFrame(summary).set_index("split"))

print("== Anotaciones por clase y split ==")
class_counts = ANN_DF.pivot_table(index="cls", columns="split", values="area", aggfunc="count").reindex(CLASS_NAMES)[list(SPLITS)]
class_counts["% train"] = (100 * class_counts["train"] / class_counts["train"].sum()).round(1)
display(class_counts)

print("== Resolución de las imágenes ==")
display(pd.Series([(im["width"], im["height"]) for c in COCO_GT.values() for im in c.imgs.values()]).value_counts().head())

# SSD300 redimensiona TODAS las imágenes a 300x300: tamaño efectivo del objeto tras el reescalado
ANN_DF["px_ssd300"] = ANN_DF["rel_size"] * 300
tr = ANN_DF[ANN_DF.split == "train"]
print("== Tamaño de objeto (train) ==")
print(tr["size_cat"].value_counts(normalize=True).round(3).to_dict())
print("sqrt(área) px originales, cuantiles:", tr["sqrt_area"].quantile([.05, .25, .5, .75, .95]).round(1).to_dict())
print(f"% objetos con tamaño efectivo < 16 px en SSD300: {100*(tr['px_ssd300'] < 16).mean():.1f}%")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(22, 4.2))

class_counts[list(SPLITS)].plot.bar(ax=axs[0]); axs[0].set_title("Anotaciones por clase"); axs[0].set_xlabel("")

for s in SPLITS:
    cnt = Counter(a["image_id"] for a in COCO_GT[s].anns.values())
    axs[1].hist([cnt.get(i, 0) for i in COCO_GT[s].imgs], bins=30, alpha=.5, label=s)
axs[1].set_title("Objetos por imagen"); axs[1].legend()

axs[2].hist(tr["sqrt_area"], bins=60); axs[2].set_xscale("log"); axs[2].set_title("√área de la caja (px originales)")
for cls in CLASS_NAMES:
    axs[3].hist(tr[tr.cls == cls]["aspect"].clip(0, 4), bins=40, alpha=.5, label=cls)
axs[3].set_title("Relación de aspecto (w/h)"); axs[3].legend(fontsize=7)
plt.tight_layout(); plt.savefig(WORK_DIR / "figs" / "eda_overview.png", dpi=130); plt.show()

# Tamaño medio por clase: ¿hay clases sistemáticamente más pequeñas?
display(tr.groupby("cls")[["sqrt_area", "px_ssd300"]].agg(["median", "min", "max"]).round(1).reindex(CLASS_NAMES))

In [ ]:
CLASS_COLORS = plt.cm.tab10(np.arange(K))

def draw_gt(coco, image_dir, image_ids, ncols=3, title_prefix=""):
    nrows = math.ceil(len(image_ids) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows)); axs = np.atleast_1d(axs).ravel()
    for ax in axs: ax.axis("off")
    for ax, iid in zip(axs, image_ids):
        info = coco.imgs[iid]; ax.imshow(Image.open(Path(image_dir) / info["file_name"]).convert("RGB"))
        for a in coco.loadAnns(coco.getAnnIds(imgIds=iid)):
            x, y, w, h = a["bbox"]
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=1.5, ec=CLASS_COLORS[CID2IDX[a["category_id"]]]))
        ax.set_title(f"{title_prefix}{info['file_name']} ({info['width']}x{info['height']})")
    handles = [patches.Patch(color=CLASS_COLORS[i], label=n) for i, n in enumerate(CLASS_NAMES)]
    fig.legend(handles=handles, loc="lower center", ncol=K); plt.tight_layout(); plt.show()

rng = random.Random(SEED)
draw_gt(COCO_GT["train"], CLEAN["train"]["images"], rng.sample(sorted(COCO_GT["train"].imgs), 3))

### ✏️ Completar con las cifras de las celdas anteriores (1 diapositiva)
- Clase más/menos frecuente y su % en train: _…_
- % de objetos *small/medium/large* y % por debajo de 16 px en SSD300: _…_
- Objetos por imagen (media/máximo) y si `MAX_DETS=300` es suficiente: _…_
- Retos visuales observados en las imágenes (oclusión, parecido entre clases, iluminación…): _…_

## 4. Infraestructura común: dataset, evaluación, tiempos y resultados

**Paso:** definir una sola vez todo lo que comparten baseline y mejoras.

**Decisiones y justificación:**
- **Dataset con tensores y cajas bien formadas.** Convierte la imagen a tensor `[0,1]` (era el error `'Image' object has no attribute 'to'`) y devuelve cajas de forma `[N,4]` también en imágenes sin objetos.
- **Aumento de datos que transforma imagen y cajas a la vez** (`DetAugment`): flips, rotaciones de 90° y color. Solo se aplica a *train*; val y test **nunca** se aumentan.
- **Evaluación con `pycocotools`** (mAP@0.5, mAP@[.5:.95], AP por tamaño y por clase, curvas PR) con el **mismo código para los dos detectores**.
  Las predicciones de cualquier modelo se guardan como lista en formato COCO; así SSD, Cascade, TTA y ensembles se evalúan y se dibujan con las mismas funciones.
- **Tiempo de inferencia** medido por imagen (batch 1, con sincronización de GPU y *warm-up*), como pide la exposición.
- **Matriz de confusión y fusión de cajas (WBF)** para el análisis de fallos y los ensembles.

In [ ]:
def rot90_boxes(boxes, W, H, k):
    """Rota k veces 90° en sentido antihorario (igual que torch.rot90(img, k, dims=(1, 2))).
    boxes: tensor [N,4] xyxy. Devuelve (boxes_nuevas, W_nueva, H_nueva)."""
    for _ in range(k % 4):
        x1, y1, x2, y2 = boxes.unbind(-1)
        boxes = torch.stack([y1, W - x2, y2, W - x1], dim=-1)   # nuevo_x = y ; nuevo_y = W - x
        W, H = H, W
    return boxes, W, H

class DetAugment:
    """Aumento de datos para detección: transforma imagen Y cajas a la vez.
    Imágenes cenitales -> flips y rotaciones de 90º no cambian la semántica de la escena.
    Probabilidades en [0,1]; 0 desactiva la transformación."""
    def __init__(self, hflip=0.0, vflip=0.0, rot90=0.0, color_jitter=0.0,
                 brightness=0.25, contrast=0.25, saturation=0.25):
        self.hflip, self.vflip, self.rot90, self.color_jitter = hflip, vflip, rot90, color_jitter
        self.b, self.c, self.s = brightness, contrast, saturation

    def __call__(self, img, target):
        _, H, W = img.shape; boxes = target["boxes"].clone()
        if random.random() < self.hflip:
            img = img.flip(-1); boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
        if random.random() < self.vflip:
            img = img.flip(-2); boxes[:, [1, 3]] = H - boxes[:, [3, 1]]
        if random.random() < self.rot90:
            k = random.choice([1, 2, 3])
            img = torch.rot90(img, k, dims=(1, 2)); boxes, W, H = rot90_boxes(boxes, W, H, k)
        if random.random() < self.color_jitter:
            img = TF.adjust_brightness(img, random.uniform(1 - self.b, 1 + self.b))
            img = TF.adjust_contrast(img, random.uniform(1 - self.c, 1 + self.c))
            img = TF.adjust_saturation(img, random.uniform(1 - self.s, 1 + self.s))
            img = img.clamp(0, 1)
        target = dict(target); target["boxes"] = boxes
        return img, target

class CocoDetDataset(Dataset):
    """Lee imágenes + anotaciones COCO ya limpias (category_id = 1..K; 0 = fondo en torchvision)."""
    def __init__(self, coco, image_dir, augment=None):
        self.coco, self.image_dir, self.augment = coco, Path(image_dir), augment
        self.ids = sorted(coco.getImgIds())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img_id = self.ids[i]; info = self.coco.imgs[img_id]
        image = TF.to_tensor(Image.open(self.image_dir / info["file_name"]).convert("RGB"))  # tensor [3,H,W] en [0,1]
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
        boxes = [[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]] for a in anns]
        labels = [a["category_id"] for a in anns]
        target = {"boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),   # reshape: imágenes sin objetos
                  "labels": torch.as_tensor(labels, dtype=torch.int64),
                  "image_id": torch.tensor([img_id])}
        if self.augment is not None:
            image, target = self.augment(image, target)
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

def make_datasets(augment=None):
    """train (con aumento opcional) / val / test (sin aumento, siempre)."""
    return {"train": CocoDetDataset(COCO_GT["train"], CLEAN["train"]["images"], augment=augment),
            "val": CocoDetDataset(COCO_GT["val"], CLEAN["val"]["images"]),
            "test": CocoDetDataset(COCO_GT["test"], CLEAN["test"]["images"])}

def make_class_balanced_sampler(ds, power=0.5):
    """Muestreo ponderado: las imágenes con clases raras se repiten más. power=1 -> inversa de la frecuencia;
    power=0.5 -> raíz (más suave). Peso de una imagen = el mayor peso de las clases que contiene."""
    freq = Counter(a["category_id"] for a in ds.coco.anns.values())
    cw = {c: (1.0 / n) ** power for c, n in freq.items()}
    weights = []
    for iid in ds.ids:
        cats = {a["category_id"] for a in ds.coco.loadAnns(ds.coco.getAnnIds(imgIds=iid))}
        weights.append(max((cw[c] for c in cats), default=min(cw.values())))
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True, generator=make_generator())

DS = make_datasets()
print({s: len(d) for s, d in DS.items()})
x, t = DS["train"][0]; print(x.shape, x.dtype, t["boxes"].shape, t["labels"][:5])

In [ ]:
METRIC_KEYS = ["mAP50_95", "mAP50", "mAP75", "AP_S", "AP_M", "AP_L"]

def evaluate_coco(coco_gt, preds, max_dets=MAX_DETS, verbose=False):
    """mAP@[.5:.95], mAP@.5, mAP@.75, AP por tamaño, AP por clase y curva PR (IoU=.5) con pycocotools.
    Mismo código y mismos umbrales para TODOS los detectores."""
    empty = {k: 0.0 for k in METRIC_KEYS}; empty.update({"per_class": {}, "pr50": None, "rec_thrs": None, "AR_max": 0.0})
    if len(preds) == 0:
        return empty
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        dt = coco_gt.loadRes([dict(p) for p in preds])      # copia: loadRes modifica los dicts
        ev = COCOeval(coco_gt, dt, "bbox")
        ev.params.maxDets = [1, 10, max_dets]
        ev.evaluate(); ev.accumulate()
    if verbose: print(buf.getvalue())
    prec = ev.eval["precision"]                              # [T(IoU), R(recall), K(clase), A(área), M(maxDets)]
    # OJO: no se usa ev.stats: con maxDets personalizado (300) pycocotools deja stats[0] (mAP@[.5:.95]) en -1.
    # Se calcula directamente de la matriz de precisión (índice -1 = max_dets; A: 0=all,1=small,2=medium,3=large).
    def ap(p):
        p = p[p > -1]; return float(p.mean()) if p.size else float("nan")
    rec = ev.eval["recall"]                                  # [T, K, A, M]
    per_class = {}
    for k, cid in enumerate(ev.params.catIds):
        per_class[CLASS_NAMES[CID2IDX[cid]]] = {"AP": ap(prec[:, :, k, 0, -1]), "AP50": ap(prec[0, :, k, 0, -1])}
    pr50 = np.where(prec[0, :, :, 0, -1] > -1, prec[0, :, :, 0, -1], np.nan)   # [R, K]
    return {"mAP50_95": ap(prec[:, :, :, 0, -1]), "mAP50": ap(prec[0, :, :, 0, -1]), "mAP75": ap(prec[5, :, :, 0, -1]),
            "AP_S": ap(prec[:, :, :, 1, -1]), "AP_M": ap(prec[:, :, :, 2, -1]), "AP_L": ap(prec[:, :, :, 3, -1]),
            "AR_max": ap(rec[:, :, 0, -1]), "per_class": per_class, "pr50": pr50, "rec_thrs": np.array(ev.params.recThrs)}

def plot_pr_curves(res, title=""):
    """Curvas Precisión-Recall (IoU=0.5) por clase, con el AP50 de cada una."""
    fig, ax = plt.subplots(figsize=(6, 5))
    for k, name in enumerate(CLASS_NAMES):
        ax.plot(res["rec_thrs"], res["pr50"][:, k], label=f"{name} (AP50={res['per_class'][name]['AP50']:.2f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.grid(alpha=.3)
    ax.set_title(f"{title}  mAP50={res['mAP50']:.3f}"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

def iou_matrix(a, b):
    """IoU entre cajas xyxy: a [N,4], b [M,4] -> [N,M]."""
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

def xywh_to_xyxy(b):
    return [b[0], b[1], b[0] + b[2], b[1] + b[3]]

def match_detections(coco_gt, preds, score_thr=0.3, iou_thr=0.5):
    """Matching voraz (por score) a un umbral de confianza fijo. Devuelve:
       conf: matriz de confusión (K+1)x(K+1) [fila=GT, columna=predicción; índice K = fondo]
       per_img: DataFrame con tp/fp/fn por imagen (para localizar las imágenes con más fallos)."""
    conf = np.zeros((K + 1, K + 1), dtype=int)
    by_img = defaultdict(list)
    for p in preds:
        if p["score"] >= score_thr: by_img[p["image_id"]].append(p)
    rows = []
    for img_id in coco_gt.getImgIds():
        gts = coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=img_id))
        gb = np.array([xywh_to_xyxy(a["bbox"]) for a in gts], dtype=float).reshape(-1, 4)
        gc = [CID2IDX[a["category_id"]] for a in gts]
        ps = sorted(by_img[img_id], key=lambda p: -p["score"])
        pb = np.array([xywh_to_xyxy(p["bbox"]) for p in ps], dtype=float).reshape(-1, 4)
        ious = iou_matrix(pb, gb) if len(ps) and len(gts) else np.zeros((len(ps), len(gts)))
        matched = np.zeros(len(gts), dtype=bool); tp = fp = 0; wrong = 0
        for i, p in enumerate(ps):
            pc = CID2IDX[p["category_id"]]; j = -1
            cand = np.where(~matched)[0]
            if len(cand):
                jj = cand[np.argmax(ious[i, cand])]
                if ious[i, jj] >= iou_thr: j = jj
            if j >= 0:
                matched[j] = True; conf[gc[j], pc] += 1
                if gc[j] == pc: tp += 1
                else: fp += 1; wrong += 1                 # caja bien localizada pero clase errónea
            else:
                conf[K, pc] += 1; fp += 1                  # falso positivo (fondo detectado como objeto)
        for j in np.where(~matched)[0]:
            conf[gc[j], K] += 1                            # falso negativo (objeto no detectado)
        rows.append({"image_id": img_id, "n_gt": len(gts), "tp": tp, "fp": fp, "fn": int((~matched).sum()) + wrong})
    return conf, pd.DataFrame(rows)

def plot_confusion(conf, title=""):
    labels = CLASS_NAMES + ["fondo"]
    norm = conf / np.maximum(conf.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(6.5, 5.5)); ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(K + 1)); ax.set_xticklabels(labels, rotation=45, ha="right"); ax.set_yticks(range(K + 1)); ax.set_yticklabels(labels)
    for i in range(K + 1):
        for j in range(K + 1):
            ax.text(j, i, conf[i, j], ha="center", va="center", color="white" if norm[i, j] > .5 else "black", fontsize=8)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real (GT)"); ax.set_title(title); plt.tight_layout(); plt.show()

def show_predictions(coco_gt, image_dir, preds, image_ids, score_thr=0.4):
    """Ejemplos cualitativos: izquierda = ground truth, derecha = predicciones con score >= score_thr."""
    by_img = defaultdict(list)
    for p in preds:
        if p["score"] >= score_thr: by_img[p["image_id"]].append(p)
    fig, axs = plt.subplots(len(image_ids), 2, figsize=(12, 6 * len(image_ids))); axs = np.atleast_2d(axs)
    for r, iid in enumerate(image_ids):
        info = coco_gt.imgs[iid]; img = Image.open(Path(image_dir) / info["file_name"]).convert("RGB")
        for c in range(2):
            axs[r, c].imshow(img); axs[r, c].axis("off")
        for a in coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=iid)):
            x, y, w, h = a["bbox"]
            axs[r, 0].add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=1.5, ec=CLASS_COLORS[CID2IDX[a["category_id"]]]))
        for p in by_img[iid]:
            x, y, w, h = p["bbox"]
            axs[r, 1].add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=1.5, ec=CLASS_COLORS[CID2IDX[p["category_id"]]]))
        axs[r, 0].set_title(f"GT - {info['file_name']}"); axs[r, 1].set_title(f"Predicciones (score >= {score_thr})")
    handles = [patches.Patch(color=CLASS_COLORS[i], label=n) for i, n in enumerate(CLASS_NAMES)]
    fig.legend(handles=handles, loc="lower center", ncol=K); plt.tight_layout(); plt.show()

# ---------- tiempos de inferencia ----------
def time_inference(predict_fn, inputs, warmup=5):
    """Latencia por imagen (batch=1, extremo a extremo desde la imagen en memoria) en ms. Sincroniza la GPU."""
    sync = torch.cuda.synchronize if DEVICE.type == "cuda" else (lambda: None)
    for x in inputs[:warmup]: predict_fn(x)
    sync(); times = []
    for x in inputs:
        t0 = time.perf_counter(); predict_fn(x); sync(); times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))

# ---------- registro de resultados (alimenta las tablas de la presentación) ----------
RESULTS = []
def log_result(experiment, model, split, metrics, ms_per_img=None, notes=""):
    global RESULTS
    RESULTS = [r for r in RESULTS if not (r["experiment"] == experiment and r["split"] == split)]   # sin duplicados al re-ejecutar
    RESULTS.append({"experiment": experiment, "model": model, "split": split,
                    **{k: metrics[k] for k in METRIC_KEYS}, "ms_per_img": ms_per_img, "notes": notes})
    save_results()

def results_df():
    return pd.DataFrame(RESULTS)

def save_results():
    results_df().to_csv(WORK_DIR / "results.csv", index=False)

# ---------- caché de predicciones ----------
def save_preds(preds, name):
    (WORK_DIR / "preds" / f"{name}.json").write_text(json.dumps(preds))

def load_preds(name):
    return json.loads((WORK_DIR / "preds" / f"{name}.json").read_text())

In [ ]:
def fuse_predictions(coco_gt, pred_lists, weights=None, iou_thr=0.55, skip_box_thr=0.001, conf_type="avg"):
    """Weighted Boxes Fusion (WBF) de varias listas de predicciones (formato COCO resultados).
    Sirve para ENSEMBLES (listas de distintos modelos) y para TTA (listas del mismo modelo con distintas transformaciones)."""
    from ensemble_boxes import weighted_boxes_fusion
    per_model = []
    for preds in pred_lists:
        d = defaultdict(list)
        for p in preds: d[p["image_id"]].append(p)
        per_model.append(d)
    out = []
    for img_id in coco_gt.getImgIds():
        W, H = coco_gt.imgs[img_id]["width"], coco_gt.imgs[img_id]["height"]
        bl, sl, ll = [], [], []
        for d in per_model:
            items = d.get(img_id, [])
            if items:
                b = np.array([[p["bbox"][0] / W, p["bbox"][1] / H, (p["bbox"][0] + p["bbox"][2]) / W,
                               (p["bbox"][1] + p["bbox"][3]) / H] for p in items], dtype=np.float32).clip(0, 1)
            else:
                b = np.zeros((0, 4), dtype=np.float32)
            bl.append(b); sl.append(np.array([p["score"] for p in items], dtype=np.float32))
            ll.append(np.array([p["category_id"] for p in items], dtype=np.float32))
        if sum(len(b) for b in bl) == 0:
            continue
        boxes, scores, labels = weighted_boxes_fusion(bl, sl, ll, weights=weights, iou_thr=iou_thr,
                                                      skip_box_thr=skip_box_thr, conf_type=conf_type)
        for b, s, l in zip(boxes, scores, labels):
            x1, y1, x2, y2 = b[0] * W, b[1] * H, b[2] * W, b[3] * H
            if x2 - x1 > 0 and y2 - y1 > 0:
                out.append({"image_id": int(img_id), "category_id": int(l), "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(s)})
    return out

## 5. Baseline 1 — SSD300-VGG16 (one-stage)

**Qué es el baseline:** *fine-tuning* de los pesos COCO **sin ninguna mejora** (sin aumento de datos, un único LR, todas las capas entrenables).
Es el punto de comparación: así, cada mejora cambia **una sola cosa** y se puede atribuir el efecto.

**Decisiones y justificación:**
- **Solo se sustituye la cabeza de clasificación** (91 clases COCO → 5 + fondo). El resto de la red conserva lo aprendido en COCO (transfer learning).
- **SGD, lr = 0.002, momentum 0.9, weight decay 5e-4:** valores habituales de la receta de torchvision para SSD. Es el optimizador estándar en detección y el que mejor se comporta en *fine-tuning* con pesos COCO.
- **15 epochs con *warm-up* y bajada del LR (×0.1) en los epochs 10 y 13:** el dataset es pequeño (~1.000 imágenes) y se parte de pesos preentrenados, así que converge pronto; el warm-up evita inestabilidad al inicio con una cabeza nueva.
  (La versión anterior bajaba el LR en el epoch 5, demasiado pronto para 15 epochs.)
- **Gradient clipping a 10** y descarte de batches con pérdida no finita: protegen el entrenamiento de picos de gradiente.
- **Se elige el mejor epoch por mAP@[.5:.95] en validación** (no por la pérdida): el mAP es la métrica que realmente nos interesa y la pérdida no siempre se correlaciona con ella.

In [ ]:
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead

def build_ssd(pretrained=PRETRAINED, input_size=None, freeze_backbone=False):
    """SSD300 con pesos COCO y cabeza de clasificación nueva para K clases (+1 fondo).
    input_size: resolución cuadrada de entrada (por defecto 300). Las anclas se adaptan al tamaño de los mapas de características.
    freeze_backbone: congela VGG + capas extra (solo se entrena la cabeza)."""
    weights = SSD300_VGG16_Weights.COCO_V1 if pretrained else None
    model = ssd300_vgg16(weights=weights, weights_backbone=None)
    old_head = model.head.classification_head
    in_channels = [layer.in_channels for layer in old_head.module_list]
    num_anchors = model.anchor_generator.num_anchors_per_location()
    model.head.classification_head = SSDClassificationHead(in_channels, num_anchors, K + 1)   # +1 = fondo
    if input_size is not None:
        model.transform.fixed_size = (input_size, input_size)
    if freeze_backbone:
        for p in model.backbone.parameters(): p.requires_grad = False
    return model

def configure_ssd_inference(model, score_thresh=SCORE_THRESH_EVAL, nms_thresh=NMS_IOU, max_dets=MAX_DETS):
    """Parámetros de postprocesado de SSD. (OJO: el valor por defecto score_thresh=0.01 del modelo ignoraba el 0.001
    que se pasaba en la versión anterior; hay que fijarlo en el propio modelo.)"""
    model.score_thresh = score_thresh; model.nms_thresh = nms_thresh
    model.detections_per_img = max_dets; model.topk_candidates = max(400, max_dets)

@torch.no_grad()
def predict_torchvision(model, dataset, batch_size=4, flip=None):
    """Predice todo el dataset -> lista de resultados COCO. flip in {None,'h','v'} para TTA (se deshace el flip en las cajas)."""
    model.eval(); preds = []
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2)
    for images, targets in loader:
        imgs = [im.to(DEVICE) for im in images]
        if flip == "h": imgs = [im.flip(-1) for im in imgs]
        if flip == "v": imgs = [im.flip(-2) for im in imgs]
        outs = model(imgs)
        for im, tgt, out in zip(images, targets, outs):
            H, W = im.shape[-2:]
            boxes = out["boxes"].cpu().numpy().copy(); scores = out["scores"].cpu().numpy(); labels = out["labels"].cpu().numpy()
            if flip == "h": boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
            if flip == "v": boxes[:, [1, 3]] = H - boxes[:, [3, 1]]
            for b, s, l in zip(boxes, scores, labels):
                if int(l) not in CID2IDX: continue          # etiqueta = category_id (1..K); 0 = fondo
                preds.append({"image_id": int(tgt["image_id"].item()), "category_id": int(l),
                              "bbox": [float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])], "score": float(s)})
    return preds

def build_optimizer(model, name="sgd", lr=0.002, weight_decay=5e-4, momentum=0.9, backbone_lr_mult=1.0):
    """backbone_lr_mult < 1 -> tasa de aprendizaje diferenciada: backbone más lento que la cabeza."""
    bb = [p for p in model.backbone.parameters() if p.requires_grad]
    bb_ids = {id(p) for p in bb}
    rest = [p for p in model.parameters() if p.requires_grad and id(p) not in bb_ids]
    groups = [{"params": rest, "lr": lr}] + ([{"params": bb, "lr": lr * backbone_lr_mult}] if bb else [])
    if name == "sgd":
        return torch.optim.SGD(groups, lr=lr, momentum=momentum, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(groups, lr=lr, weight_decay=weight_decay)
    raise ValueError(name)

def make_scheduler(optimizer, iters_per_epoch, decay_epochs=(10, 13), gamma=0.1, warmup_iters=200):
    """Warm-up lineal + decaimiento escalonado, actualizado POR ITERACIÓN (scheduler.step() tras cada optimizer.step())."""
    def lr_lambda(it):
        warm = 0.001 + (1 - 0.001) * it / warmup_iters if it < warmup_iters else 1.0
        return warm * gamma ** sum(it >= e * iters_per_epoch for e in decay_epochs)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def train_one_epoch(model, loader, optimizer, scheduler, clip=10.0):
    model.train(); losses = []
    for images, targets in loader:
        images = [im.to(DEVICE) for im in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss = sum(model(images, targets).values())
        if not torch.isfinite(loss):
            print("  [aviso] pérdida no finita, se salta el batch"); optimizer.zero_grad(); scheduler.step(); continue
        optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step(); scheduler.step(); losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else float("nan")

@torch.no_grad()
def mean_detection_loss(model, loader):
    """Pérdida en validación (torchvision solo la calcula en modo train; SSD no tiene BN ni dropout, así que es seguro)."""
    model.train(); vals = []
    for images, targets in loader:
        images = [im.to(DEVICE) for im in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        vals.append(float(sum(model(images, targets).values()).cpu()))
    return float(np.mean(vals))

def load_ssd(name):
    ck = torch.load(WORK_DIR / "ckpt" / f"{name}.pth", map_location=DEVICE)
    model = build_ssd(pretrained=False, **ck["build"]).to(DEVICE); model.load_state_dict(ck["state"]); model.eval()
    configure_ssd_inference(model); return model

def run_ssd_experiment(name, epochs=15, lr=0.002, optimizer="sgd", weight_decay=5e-4, batch_size=4,
                       augment=None, freeze_backbone=False, backbone_lr_mult=1.0, balanced_sampler=False,
                       input_size=None, decay_epochs=(10, 13), warmup_iters=200, force_retrain=False, notes=""):
    """Entrena (o recarga) SSD300, elige el mejor epoch por mAP@[.5:.95] en VAL, evalúa val y test, mide tiempo y registra resultados.
    augment: dict para DetAugment (p. ej. dict(hflip=.5, vflip=.5)) o None (= baseline)."""
    seed_everything()
    ckpt = WORK_DIR / "ckpt" / f"{name}.pth"; hist_path = WORK_DIR / "preds" / f"{name}_history.csv"
    build_args = dict(input_size=input_size, freeze_backbone=freeze_backbone)
    ds = make_datasets(DetAugment(**augment) if augment else None)
    model = build_ssd(**build_args).to(DEVICE); configure_ssd_inference(model)

    if ckpt.exists() and not force_retrain:
        print(f"[{name}] checkpoint encontrado -> se omite el entrenamiento (force_retrain=True para repetirlo)")
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["state"])
        history = pd.read_csv(hist_path) if hist_path.exists() else pd.DataFrame()
    else:
        sampler = make_class_balanced_sampler(ds["train"]) if balanced_sampler else None
        train_loader = DataLoader(ds["train"], batch_size=batch_size, shuffle=sampler is None, sampler=sampler, collate_fn=collate_fn,
                                  num_workers=2, generator=make_generator(), worker_init_fn=seed_worker, drop_last=True)
        val_loader = DataLoader(ds["val"], batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2)
        opt = build_optimizer(model, optimizer, lr, weight_decay, backbone_lr_mult=backbone_lr_mult)
        sched = make_scheduler(opt, len(train_loader), decay_epochs, warmup_iters=warmup_iters)
        history, best = [], -1.0
        for ep in range(1, epochs + 1):
            t0 = time.time()
            tr = train_one_epoch(model, train_loader, opt, sched); vl = mean_detection_loss(model, val_loader)
            m = evaluate_coco(COCO_GT["val"], predict_torchvision(model, ds["val"]))
            history.append({"epoch": ep, "train_loss": tr, "val_loss": vl, "val_mAP50": m["mAP50"], "val_mAP50_95": m["mAP50_95"]})
            print(f"[{name}] ep {ep:02d}/{epochs} train={tr:.3f} val={vl:.3f} val mAP50={m['mAP50']:.3f} mAP50:95={m['mAP50_95']:.3f} ({time.time()-t0:.0f}s)")
            if m["mAP50_95"] > best:
                best = m["mAP50_95"]; torch.save({"state": model.state_dict(), "build": build_args}, ckpt)
        history = pd.DataFrame(history); history.to_csv(hist_path, index=False)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["state"])    # mejor epoch según val

    model.eval(); configure_ssd_inference(model)
    out = {"name": name, "history": history}
    timing_imgs = [ds["test"][i][0] for i in range(min(50, len(ds["test"])))]
    ms, ms_std = time_inference(lambda im: model([im.to(DEVICE)]), timing_imgs)
    for split in ("val", "test"):
        preds = predict_torchvision(model, ds[split]); save_preds(preds, f"{name}_{split}")
        out[split] = evaluate_coco(COCO_GT[split], preds); out[f"preds_{split}"] = preds
        log_result(name, "SSD300-VGG16", split, out[split], ms, notes)
        print(f"[{name}] {split}: mAP50={out[split]['mAP50']:.3f}  mAP50:95={out[split]['mAP50_95']:.3f}  ({ms:.1f}±{ms_std:.1f} ms/img)")
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

def plot_history(history, title=""):
    if len(history) == 0: return
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
    history.plot(x="epoch", y=["train_loss", "val_loss"], ax=axs[0], grid=True, title=f"{title} - pérdida")
    history.plot(x="epoch", y=["val_mAP50", "val_mAP50_95"], ax=axs[1], grid=True, title=f"{title} - mAP en val")
    plt.tight_layout(); plt.show()

In [ ]:
# ===== BASELINE SSD300: SGD lr=0.002, 15 epochs, sin aumento, todo entrenable =====
res_ssd_base = run_ssd_experiment("ssd_baseline", epochs=15, lr=0.002, optimizer="sgd",
                                  notes="baseline: sin aumento, SGD, todo entrenable")

In [ ]:
plot_history(res_ssd_base["history"], "SSD300 baseline")
plot_pr_curves(res_ssd_base["val"], "SSD300 baseline (val)")
display(pd.DataFrame(res_ssd_base["test"]["per_class"]).T.round(3).rename(columns={"AP": "AP@[.5:.95]"}))   # AP por clase (test)

### ✏️ Completar — lectura del baseline SSD300
- ¿Hay sobreajuste? (compara `train_loss` y `val_loss`; ¿en qué epoch se estanca el mAP de val?): _…_
- ¿Qué clases fallan más y por qué? (AP por clase + tamaños del análisis exploratorio): _…_
- ¿Qué indican `AP_S / AP_M / AP_L` sobre el efecto del reescalado a 300×300?: _…_

## 6. Baseline 2 — Cascade R-CNN R50-FPN (two-stage, Detectron2)

**Configuración:** `Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml` con `MASK_ON=False` (solo cajas, como pide el guion) y pesos COCO.

**Decisiones y justificación:**
- **Aumento de datos desactivado en el baseline** (`RANDOM_FLIP="none"` y una sola escala). Detectron2 aplica por defecto flip horizontal y multi-escala; si se dejaran,
  el baseline de Cascade tendría aumento y el de SSD no, y la comparación inicial no sería justa. Reactivarlos es precisamente la mejora E4.
- **Lote 4 y lr = 0.005:** regla de escalado lineal de Detectron2 (0.02 para lote 16 → 0.005 para lote 4). El 0.00025 de la versión anterior es demasiado bajo para este lote y haría el ajuste muy lento.
  Si hay falta de memoria en la T4: `ims_per_batch=2, lr=0.0025`.
- **10 epochs** (nº de iteraciones calculado a partir del tamaño de train) con bajada del LR al 70 % y 90 %, la receta estándar de Detectron2.
- **Evaluación con el mismo `evaluate_coco` que SSD**, usando `DefaultPredictor` con el mismo umbral de score, NMS y máximo de detecciones. El `COCOEvaluator` interno solo sirve para ver la evolución durante el entrenamiento.
- **Precisión mixta (AMP)** para acelerar el entrenamiento en la T4 sin cambiar el resultado de forma apreciable.

In [ ]:
def register_d2_datasets():
    for split in SPLITS:
        name = f"{DATASET_NAME}_{split}"
        if name in DatasetCatalog.list():
            DatasetCatalog.remove(name); MetadataCatalog.remove(name)
        register_coco_instances(name, {}, str(CLEAN[split]["json"]), str(CLEAN[split]["images"]))
        MetadataCatalog.get(name).set(thing_classes=CLASS_NAMES)

class CocoTrainer(DefaultTrainer):
    """Trainer con evaluación COCO periódica sobre val."""
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        return COCOEvaluator(dataset_name, output_dir=output_folder or os.path.join(cfg.OUTPUT_DIR, "inference"), distributed=False)

class CocoTrainerAug(CocoTrainer):
    """Aumento geométrico + color en entrenamiento (flips H y V, brillo, contraste, saturación)."""
    @classmethod
    def build_train_loader(cls, cfg):
        augs = [T.ResizeShortestEdge(cfg.INPUT.MIN_SIZE_TRAIN, cfg.INPUT.MAX_SIZE_TRAIN, "choice"),
                T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
                T.RandomFlip(prob=0.5, horizontal=False, vertical=True),
                T.RandomBrightness(0.8, 1.2), T.RandomContrast(0.8, 1.2), T.RandomSaturation(0.8, 1.2)]
        return build_detection_train_loader(cfg, mapper=DatasetMapper(cfg, is_train=True, augmentations=augs))

def make_cascade_predictor(cfg, weights):
    """Predictor con los MISMOS parámetros de postprocesado que SSD."""
    c = cfg.clone(); c.MODEL.WEIGHTS = str(weights)
    c.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_EVAL
    c.MODEL.ROI_HEADS.NMS_THRESH_TEST = NMS_IOU
    c.TEST.DETECTIONS_PER_IMAGE = MAX_DETS
    return DefaultPredictor(c)

def predict_detectron2(predictor, split):
    import cv2
    coco = COCO_GT[split]; preds = []
    for iid in sorted(coco.getImgIds()):
        bgr = cv2.imread(str(Path(CLEAN[split]["images"]) / coco.imgs[iid]["file_name"]))   # Detectron2 espera BGR
        inst = predictor(bgr)["instances"].to("cpu")
        for b, s, c in zip(inst.pred_boxes.tensor.numpy(), inst.scores.numpy(), inst.pred_classes.numpy()):
            preds.append({"image_id": int(iid), "category_id": CLASS_IDS[int(c)],         # índice 0..K-1 -> category_id
                          "bbox": [float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])], "score": float(s)})
    return preds

def run_cascade_experiment(name, epochs=10, lr=0.005, ims_per_batch=4, flip="none", min_size_train=(800,),
                           aug_geom_color=False, amp=True, extra_cfg=None, force_retrain=False, notes=""):
    """Entrena (o recarga) Cascade R-CNN, evalúa val y test con evaluate_coco, mide tiempo y registra resultados.
    lr=0.005 con batch 4 sigue la regla de escalado lineal de Detectron2 (0.02 para batch 16). Si hay OOM: ims_per_batch=2, lr=0.0025.
    extra_cfg: dict {'CLAVE.ANIDADA': valor} para tocar cualquier parámetro de la configuración."""
    seed_everything(); register_d2_datasets()
    cfg_name = "Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(cfg_name))
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(cfg_name)          # pesos COCO
    cfg.MODEL.MASK_ON = False                                            # solo cajas
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = K
    cfg.DATASETS.TRAIN = (f"{DATASET_NAME}_train",); cfg.DATASETS.TEST = (f"{DATASET_NAME}_val",)
    cfg.DATALOADER.NUM_WORKERS = 2; cfg.SEED = SEED
    n_train = len({a["image_id"] for a in COCO_GT["train"].anns.values()})          # Detectron2 descarta imágenes vacías
    ipe = math.ceil(n_train / ims_per_batch)
    cfg.SOLVER.IMS_PER_BATCH = ims_per_batch; cfg.SOLVER.BASE_LR = lr; cfg.SOLVER.MAX_ITER = epochs * ipe
    cfg.SOLVER.STEPS = (int(0.7 * cfg.SOLVER.MAX_ITER), int(0.9 * cfg.SOLVER.MAX_ITER))
    cfg.SOLVER.WARMUP_ITERS = min(500, cfg.SOLVER.MAX_ITER // 10)
    cfg.SOLVER.CHECKPOINT_PERIOD = 10**9          # solo se guarda model_final (los checkpoints pesan ~500 MB)
    cfg.SOLVER.AMP.ENABLED = amp
    cfg.TEST.EVAL_PERIOD = 2 * ipe                # evaluación en val cada 2 epochs (curva de evolución)
    cfg.INPUT.RANDOM_FLIP = flip                  # "none" | "horizontal" | "vertical"
    cfg.INPUT.MIN_SIZE_TRAIN = tuple(min_size_train)
    for k, v in (extra_cfg or {}).items():
        node = cfg; *path, last = k.split(".")
        for part in path:
            node = getattr(node, part)
        setattr(node, last, v)
    cfg.OUTPUT_DIR = str(WORK_DIR / "d2" / name); os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    final = Path(cfg.OUTPUT_DIR) / "model_final.pth"
    (Path(cfg.OUTPUT_DIR) / "config.yaml").write_text(cfg.dump())          # reproducibilidad

    if final.exists() and not force_retrain:
        print(f"[{name}] model_final.pth encontrado -> se omite el entrenamiento")
    else:
        trainer = (CocoTrainerAug if aug_geom_color else CocoTrainer)(cfg)
        trainer.resume_or_load(resume=False); trainer.train()
        del trainer; gc.collect(); torch.cuda.empty_cache()

    predictor = make_cascade_predictor(cfg, final)
    import cv2
    test_imgs = [cv2.imread(str(Path(CLEAN["test"]["images"]) / COCO_GT["test"].imgs[i]["file_name"])) for i in sorted(COCO_GT["test"].getImgIds())[:50]]
    ms, ms_std = time_inference(lambda im: predictor(im), test_imgs)
    out = {"name": name, "cfg": cfg}
    for split in ("val", "test"):
        preds = predict_detectron2(predictor, split); save_preds(preds, f"{name}_{split}")
        out[split] = evaluate_coco(COCO_GT[split], preds); out[f"preds_{split}"] = preds
        log_result(name, "Cascade R-CNN R50-FPN", split, out[split], ms, notes)
        print(f"[{name}] {split}: mAP50={out[split]['mAP50']:.3f}  mAP50:95={out[split]['mAP50_95']:.3f}  ({ms:.1f}±{ms_std:.1f} ms/img)")
    del predictor; gc.collect(); torch.cuda.empty_cache()
    return out

In [ ]:
# ===== BASELINE Cascade R-CNN: SGD lr=0.005, batch 4, 10 epochs, SIN aumento (flip none, una escala) =====
res_cas_base = run_cascade_experiment("cascade_baseline", epochs=10, lr=0.005, flip="none", min_size_train=(800,),
                                      notes="baseline: sin aumento, SGD, todo entrenable")

In [ ]:
plot_pr_curves(res_cas_base["val"], "Cascade R-CNN baseline (val)")
display(pd.DataFrame(res_cas_base["test"]["per_class"]).T.round(3).rename(columns={"AP": "AP@[.5:.95]"}))

## 7. Comparativa de los baselines

**Paso:** tabla y curvas PR de ambos baselines en las mismas condiciones. Esta comparación es la que guía las mejoras de la sección siguiente.

In [ ]:
def summary_table(splits=("val", "test"), experiments=None):
    df = results_df()
    if experiments is not None: df = df[df.experiment.isin(experiments)]
    df = df[df.split.isin(splits)].copy()
    cols = ["mAP50", "mAP50_95", "mAP75", "AP_S", "AP_M", "AP_L"]
    df[cols] = df[cols].astype(float).round(3)
    return df[["experiment", "model", "split"] + cols + ["ms_per_img", "notes"]].sort_values(["split", "experiment"]).reset_index(drop=True)

display(summary_table(experiments=["ssd_baseline", "cascade_baseline"]))

# Superposición de curvas PR (IoU=.5, media entre clases) de ambos baselines en val
fig, ax = plt.subplots(figsize=(6, 4.5))
for r, lab in [(res_ssd_base, "SSD300"), (res_cas_base, "Cascade R-CNN")]:
    ax.plot(r["val"]["rec_thrs"], np.nanmean(r["val"]["pr50"], axis=1), label=f"{lab} (mAP50={r['val']['mAP50']:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision media (IoU=.5)"); ax.grid(alpha=.3); ax.legend(); plt.show()

### ✏️ Completar — conclusiones del baseline (diapositiva 2)
1. ¿Quién gana y por cuánto en mAP@0.5 y mAP@[.5:.95]? _…_
2. ¿Qué coste en ms/imagen tiene cada uno? _…_
3. ¿Qué hipótesis de mejora se derivan de los fallos observados? _…_

## 8. Estrategias de mejora

**Criterio general:** cada mejora cambia **una cosa** respecto al baseline y está motivada por algo medido en el análisis exploratorio (no ensayo y error).
Las decisiones se toman con **validación**; el test solo se mira en la tabla final. Cuando una mejora no ayuda, se anota igualmente (es un resultado válido).

| ID | Mejora | Hipótesis (por qué debería ayudar) | Dato que la motiva |
|---|---|---|---|
| E1 | Aumento de datos en SSD (flips H/V, rot. 90°, color) | La vista es cenital: giros y volteos generan escenas realistas; con ~1.000 imágenes, regularizar reduce el sobreajuste | Imágenes cenitales, dataset pequeño |
| E2 | LR diferenciado en SSD (backbone ×0.1) | El backbone ya está bien entrenado en COCO y solo la cabeza es nueva: no conviene "destrozar" sus pesos | Cabeza nueva + backbone preentrenado |
| E3 | Combinación de E1 y E2 (solo si ambas mejoran en val) | Si las dos ayudan por separado y actúan sobre cosas distintas (datos / optimización), pueden sumar | Resultados de E1 y E2 |
| E4 | Aumento de datos en Cascade (flips H/V, color, multi-escala) | Misma hipótesis que E1, y permite comparar el efecto del aumento en un detector one-stage frente a uno two-stage | Igual que E1 |
| E5 | TTA (flips) en el mejor SSD | Promediar predicciones de varias vistas reduce la varianza y recupera objetos que una sola vista pierde | Vista cenital (flips válidos) |
| E6 | Ensemble SSD + Cascade (WBF) | Un detector one-stage y uno two-stage cometen errores distintos; fusionar sus cajas mejora robustez y cobertura | Resultados de los baselines |

### E1 — Aumento de datos en SSD
**Cambio:** `DetAugment` con probabilidad 0.5 en flip horizontal, flip vertical, rotación de 90° y variación de brillo/contraste/saturación.
**Justificación:** las transformaciones geométricas son válidas porque en una vista cenital la orientación de la bandeja no aporta información de la clase;
el color simula cambios de iluminación. **Se evita** el recorte agresivo, que podría cortar objetos y dejar anotaciones inconsistentes.

In [ ]:
res_e1 = run_ssd_experiment("ssd_E1_aug", augment=dict(hflip=.5, vflip=.5, rot90=.5, color_jitter=.5),
                            notes="E1: aumento geométrico + color")
plot_history(res_e1["history"], "E1")

### E2 — Estrategia de transfer learning: LR diferenciado
**Cambio:** el backbone (VGG16 + capas extra) se entrena con **0.1× el LR** de la cabeza (0.0002 frente a 0.002).
**Justificación:** la cabeza de clasificación es nueva (inicialización aleatoria) y necesita aprender rápido; el backbone ya extrae buenas características de COCO y un LR alto lo perturbaría.
Es un punto intermedio entre "entrenar todo igual" (baseline) y "congelar el backbone", que sería demasiado rígido porque este dominio (lechugas cenitales) difiere bastante de COCO.

In [ ]:
res_e2 = run_ssd_experiment("ssd_E2_diff_lr", backbone_lr_mult=0.1, notes="E2: LR backbone = 0.1x")
plot_history(res_e2["history"], "E2")

### E3 — Combinación de E1 y E2
**Decisión automática y justificada:** solo se combinan si **ambas mejoran en validación** respecto al baseline; combinar una mejora que no ayuda
podría empeorar el resultado y no sabríamos por qué. La decisión se toma con el mAP@[.5:.95] de val.

In [ ]:
def val_map(name):
    df = results_df(); df = df[(df.split == "val") & (df.experiment == name)]
    return float(df.iloc[0]["mAP50_95"]) if len(df) else float("nan")

def best_by_val(names):
    """Experimento con mejor mAP@[.5:.95] en VALIDACIÓN (el test no interviene en la elección)."""
    return max(names, key=lambda n: -1 if np.isnan(val_map(n)) else val_map(n))

base = val_map("ssd_baseline")
use_aug, use_lr = val_map("ssd_E1_aug") > base, val_map("ssd_E2_diff_lr") > base
print(f"val mAP50:95  baseline={base:.3f}  E1={val_map('ssd_E1_aug'):.3f}  E2={val_map('ssd_E2_diff_lr'):.3f}")
if use_aug and use_lr:
    res_e3 = run_ssd_experiment("ssd_E3_combo", augment=dict(hflip=.5, vflip=.5, rot90=.5, color_jitter=.5),
                                backbone_lr_mult=0.1, notes="E3: E1 + E2")
    plot_history(res_e3["history"], "E3")
else:
    print("E3 omitido: E1 y E2 no mejoran ambas al baseline en val (se explica en la memoria).")

### E4 — Aumento de datos en Cascade R-CNN
**Cambio:** flips horizontal y vertical, variaciones de color y **multi-escala** (lado corto entre 640 y 800 px).
**Justificación:** misma hipótesis que E1 (para poder comparar one-stage vs two-stage); la multi-escala añade robustez al tamaño del objeto y es la práctica estándar de Detectron2.

In [ ]:
res_e4 = run_cascade_experiment("cascade_E4_aug", aug_geom_color=True, min_size_train=(640, 672, 704, 736, 768, 800),
                                notes="E4: flips H+V + color + multiescala")

### Elección de los mejores modelos (en validación)
**Decisión:** el "mejor SSD" y el "mejor Cascade" se eligen **automáticamente por mAP@[.5:.95] en validación**. Se usarán en TTA, ensemble y análisis cualitativo.

In [ ]:
SSD_NAMES = [n for n in ["ssd_baseline", "ssd_E1_aug", "ssd_E2_diff_lr", "ssd_E3_combo"] if (WORK_DIR / "ckpt" / f"{n}.pth").exists()]
CAS_NAMES = ["cascade_baseline", "cascade_E4_aug"]
BEST_SSD, BEST_CASCADE = best_by_val(SSD_NAMES), best_by_val(CAS_NAMES)
print("Mejor SSD     :", BEST_SSD, f"(val mAP50:95 = {val_map(BEST_SSD):.3f})")
print("Mejor Cascade :", BEST_CASCADE, f"(val mAP50:95 = {val_map(BEST_CASCADE):.3f})")
def ms_of(name):
    df = results_df(); return float(df[(df.experiment == name) & (df.split == "test")].iloc[0]["ms_per_img"])

### E5 — Test-time augmentation (TTA) en el mejor SSD
**Cambio:** se predice la imagen original + volteada horizontal + volteada vertical, se deshacen los volteos en las cajas y se **fusionan con WBF** (Weighted Boxes Fusion).
**Justificación:** WBF promedia las cajas coincidentes (en vez de descartar las de menor score como hace el NMS), lo que suele mejorar la localización.
**Coste:** 3 pasadas por imagen → el tiempo de inferencia se triplica aproximadamente (se anota en la tabla como estimación); hay que valorar si el aumento de mAP lo compensa.

In [ ]:
model = load_ssd(BEST_SSD); ds_eval = make_datasets()
for split in ("val", "test"):
    plain = predict_torchvision(model, ds_eval[split])
    ph = predict_torchvision(model, ds_eval[split], flip="h"); pv = predict_torchvision(model, ds_eval[split], flip="v")
    fused = fuse_predictions(COCO_GT[split], [plain, ph, pv], iou_thr=0.55); save_preds(fused, f"ssd_E5_tta_{split}")
    m = evaluate_coco(COCO_GT[split], fused)
    log_result("ssd_E5_tta", "SSD300-VGG16", split, m, 3 * ms_of(BEST_SSD), f"E5: {BEST_SSD} + TTA flips (WBF); tiempo estimado = 3 pasadas")
    print(f"TTA {split}: mAP50={m['mAP50']:.3f}  mAP50:95={m['mAP50_95']:.3f}")
del model; gc.collect(); torch.cuda.empty_cache()

### E6 — Ensemble SSD + Cascade con WBF
**Cambio:** se fusionan con WBF las predicciones del mejor SSD y del mejor Cascade.
**Justificación:** son arquitecturas muy distintas (one-stage con anclas frente a two-stage en cascada) y por tanto **cometen errores distintos**; fusionarlas combina sus aciertos.
**Decisión:** los **pesos del ensemble** (1:1, 1:2, 1:3) y el **umbral de IoU de fusión** (0.5, 0.6, 0.7) se eligen **en validación**; el test solo se evalúa con la configuración elegida.
El tiempo de inferencia del ensemble es aproximadamente la suma de ambos detectores.

In [ ]:
best = None
for split in ("val", "test"):
    p_ssd, p_cas = load_preds(f"{BEST_SSD}_{split}"), load_preds(f"{BEST_CASCADE}_{split}")
    if split == "val":
        for w in ([1, 1], [1, 2], [1, 3]):
            for thr in (0.5, 0.6, 0.7):
                m = evaluate_coco(COCO_GT["val"], fuse_predictions(COCO_GT["val"], [p_ssd, p_cas], weights=w, iou_thr=thr))
                if best is None or m["mAP50_95"] > best[0]: best = (m["mAP50_95"], w, thr)
        print(f"Mejor combinación en val: pesos={best[1]}, iou_thr={best[2]} (mAP50:95={best[0]:.3f})")
    fused = fuse_predictions(COCO_GT[split], [p_ssd, p_cas], weights=best[1], iou_thr=best[2]); save_preds(fused, f"ensemble_E6_{split}")
    m = evaluate_coco(COCO_GT[split], fused)
    log_result("ensemble_E6", "SSD300 + Cascade (WBF)", split, m, ms_of(BEST_SSD) + ms_of(BEST_CASCADE),
               f"E6: {BEST_SSD} + {BEST_CASCADE}, pesos={best[1]}, iou={best[2]}; tiempo estimado = suma")
    print(f"Ensemble {split}: mAP50={m['mAP50']:.3f}  mAP50:95={m['mAP50_95']:.3f}")

### ✏️ Completar — resultado de cada mejora (una frase por experimento)
Para cada E1–E6: ¿mejoró? ¿cuánto (Δ mAP@0.5 y Δ mAP@[.5:.95] respecto al baseline correspondiente)? ¿Se confirmó la hipótesis? Si no mejoró, ¿por qué crees que no? _…_

## 9. Resultados finales y análisis de fallos

**Paso:** tabla completa en **test** (con la elección de modelos ya hecha en val), evolución del mAP, relación precisión/velocidad,
y análisis cualitativo de errores (matrices de confusión + ejemplos). Es el contenido de la diapositiva 4 de la exposición.

In [ ]:
df_final = summary_table(splits=("test",))
display(df_final)

val_tab = summary_table(splits=("val",))
print("Referencia en validación (donde se tomaron las decisiones):"); display(val_tab[["experiment", "mAP50", "mAP50_95", "ms_per_img"]])

# Gráfica de evolución: baseline -> cada mejora (test)
t = results_df(); t = t[t.split == "test"].copy(); t["fam"] = t.model.str.split().str[0]
fig, ax = plt.subplots(figsize=(11, 4.5)); order = t.sort_values("mAP50_95")
ax.barh(order.experiment, order.mAP50_95.astype(float), color=["#4C78A8" if f.startswith("SSD") else "#F58518" for f in order.fam])
ax.set_xlabel("mAP@[.5:.95] (test)"); ax.grid(axis="x", alpha=.3); plt.tight_layout()
plt.savefig(WORK_DIR / "figs" / "evolucion_map.png", dpi=130); plt.show()

# mAP frente a tiempo de inferencia (trade-off velocidad/precisión)
tt = t.dropna(subset=["ms_per_img"])
fig, ax = plt.subplots(figsize=(6, 4.5)); ax.scatter(tt.ms_per_img.astype(float), tt.mAP50_95.astype(float))
for _, r in tt.iterrows(): ax.annotate(r.experiment, (r.ms_per_img, r.mAP50_95), fontsize=7)
ax.set_xlabel("ms / imagen"); ax.set_ylabel("mAP@[.5:.95] (test)"); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# Análisis de fallos en TEST con umbral de confianza 0.4 (para visualizar se usa un umbral razonable; para el mAP se usó 0.001)
CANDIDATES = {"SSD baseline": "ssd_baseline_test", "Cascade baseline": "cascade_baseline_test",
              f"Mejor SSD ({BEST_SSD})": f"{BEST_SSD}_test", f"Mejor Cascade ({BEST_CASCADE})": f"{BEST_CASCADE}_test",
              "Ensemble": "ensemble_E6_test"}
for label, key in CANDIDATES.items():
    conf, per_img = match_detections(COCO_GT["test"], load_preds(key), score_thr=0.4, iou_thr=0.5)
    plot_confusion(conf, f"{label} (test)")
    print(f"{label}: aciertos(TP)={per_img.tp.sum()}  falsos positivos(FP)={per_img.fp.sum()}  no detectados(FN)={per_img.fn.sum()}")

# Ejemplos visuales con el ensemble: 2 imágenes con más errores + 2 aleatorias
preds = load_preds("ensemble_E6_test")
_, per_img = match_detections(COCO_GT["test"], preds, score_thr=0.4)
worst = per_img.assign(err=per_img.fp + per_img.fn).sort_values("err", ascending=False).image_id.head(2).tolist()
ok = random.Random(SEED).sample(sorted(COCO_GT["test"].getImgIds()), 2)
show_predictions(COCO_GT["test"], CLEAN["test"]["images"], preds, worst + ok, score_thr=0.4)

### ✏️ Completar — análisis comparativo (diapositiva 4)
- **Mejora acumulada:** ¿qué técnica aportó más? ¿cuál no funcionó y por qué? _…_
- **Velocidad vs precisión:** ms/imagen de cada configuración; ¿merecen la pena TTA y el ensemble? _…_
- **One-stage vs two-stage:** ¿en qué clases/tamaños gana cada uno? (AP por clase y AP_S/M/L) _…_
- **Fallos frecuentes (con las matrices de confusión y las imágenes de arriba):** confusiones entre clases, falsos positivos, objetos no detectados (¿pequeños? ¿ocluidos?) _…_
- **Limitaciones:** una sola semilla, dataset pequeño, etc. _…_

### Antes de entregar (`GRUPO_N.zip`)
- [ ] Tercer integrante en la cabecera.
- [ ] *Restart & Run all* sin errores y con las salidas visibles.
- [ ] Rellenadas las celdas "✏️ Completar".
- [ ] Zip con: diapositivas en PDF, este cuaderno, `README.md` y `results.csv`.
- [ ] Fecha límite: grupo 1 → lunes 30/11 11:30 · grupo 2 → jueves 26/11 16:30.

In [ ]:
save_results()
print("Resultados guardados en", WORK_DIR / "results.csv")
display(results_df().round(3))